# 03 — Profiling & Silver Cleaning.


O Data Profiling vai ajudar-nos a entender a qualidade dos dados (nulos, duplicados e distribuição). A Silver Cleaning aplicará as regras de negócio para corrigir esses problemas e ajustar os esquemas (schema casting).


### 3.1 Setup inicial
Nesta fase vamos realizar os imports, carregar as tabelas previamente guardadas em formato Delta e definir os caminhos base de cada uma delas.


In [0]:
# Setup inicial
# imports necessários, definição de caminhos base necessários e lista com as tabelas a trabalhar
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Caminhos Delta necessários para correr este notebook de forma independente
bronze_delta_path = "/Volumes/main/default/faers_data/delta/bronze"
silver_delta_path = "/Volumes/main/default/faers_data/delta/silver"

tables = ["demo", "drug", "reac", "outc"]





### 3.1.1 Funções Auxiliares para Profiling



In [0]:
%run ./utils/data_quality_helpers

In [0]:
# Primeiro carregamos as tabelas Delta da camada Bronze
bronze_dfs = {}
for table in tables:
    bronze_dfs[table] = spark.read.format("delta").load(f"{bronze_delta_path}/{table}/")


In [0]:
# visualização das tabelas
for table in tables:
    print(f"\nTabela: {table}")
    display(bronze_dfs[table].limit(5))


### 3.2 Schema Casting (Conversão de Tipos de Dados)

Na camada Bronze, todas as colunas foram ingeridas temporariamente como `string` para garantir a fidelidade aos ficheiros originais. Agora, na camada Silver, é necessário atribuir os tipos de dados semânticos corretos (Datas, Números Inteiros e Decimais).

**Principais Transformações:**
1. **Datas:** Os ficheiros FAERS utilizam o formato `AAAAMMDD`. Colunas como `event_dt` ou `fda_dt` serão convertidas para `DateType`.*
2. **Métricas Clínicas e Doses:** Colunas quantitativas como `age` (idade), `wt` (peso) e `dose_amt` (quantidade da dose) serão convertidas para `DoubleType` para permitir agregações/cálculos matemáticos (médias, distribuições) na camada Gold.
3. As variáveis categóricas e identificadores (como `primaryid`, `pt`, `outc_cod`) mantêm-se como `string`.


In [0]:
# Contagem de nulos ANTES do casting (Bronze)
print("=== CONTAGEM DE NULOS - ANTES DO SCHEMA CASTING ===")
print("(Dados ainda em formato string na camada Bronze)\n")

null_counts_before = analisar_nulos(
    dfs_dict=bronze_dfs,
    return_snapshots=True
)

print("✅ Snapshot guardado em 'null_counts_before'\n")

In [0]:

from pyspark.sql.types import DoubleType

silver_dfs = {}
print("=== SCHEMA CASTING ===\n")

for table_name, df in bronze_dfs.items():
    df_cast = df
    # Transformações específicas para a tabela DEMO
    if table_name == "demo":
        df_cast = df_cast.withColumn("event_dt", F.to_date(F.col("event_dt"), "yyyyMMdd")) \
                         .withColumn("mfr_dt", F.to_date(F.col("mfr_dt"), "yyyyMMdd")) \
                         .withColumn("init_fda_dt", F.to_date(F.col("init_fda_dt"), "yyyyMMdd")) \
                         .withColumn("fda_dt", F.to_date(F.col("fda_dt"), "yyyyMMdd")) \
                         .withColumn("rept_dt", F.to_date(F.col("rept_dt"), "yyyyMMdd")) \
                         .withColumn("age", F.col("age").cast(DoubleType())) \
                         .withColumn("wt", F.col("wt").cast(DoubleType()))                    
    # Transformações específicas para a tabela DRUG
    elif table_name == "drug":
        df_cast = df_cast.withColumn("exp_dt", F.to_date(F.col("exp_dt"), "yyyyMMdd")) \
                         .withColumn("dose_amt", F.col("dose_amt").cast(DoubleType())) \
                         .withColumn("cum_dose_chr", F.col("cum_dose_chr").cast(DoubleType()))
                         
    # As tabelas REAC e OUTC contêm apenas identificadores e códigos em texto, 
    # pelo que não necessitam de casting numérico/temporal.
    
    silver_dfs[table_name] = df_cast
    print(f"✅ Schema atualizado para a tabela: {table_name.upper()}")

print("\n✅ Schema casting concluído!")
print("📌 Tabelas guardadas no dicionário 'silver_dfs' (em memória)")
print("📌 Escrita para Delta será feita APÓS todas as transformações de limpeza")

In [0]:
# Contagem de nulos DEPOIS do casting (Silver) e comparação
print("\n=== CONTAGEM DE NULOS - DEPOIS DO SCHEMA CASTING ===")
print("(Dados convertidos para tipos semânticos na camada Silver)\n")

null_counts_after = analisar_nulos(
    dfs_dict=silver_dfs,
    return_snapshots=True
)

print("✅ Snapshot guardado em 'null_counts_after'\n")

# Comparar antes vs depois
data_loss = comparar_nulos(
    snapshot_before=null_counts_before,
    snapshot_after=null_counts_after,
    table_list=tables
)


### 3.3 Profiling inicial
Após verificarmos que as tabelas bronze foram corretamente carregadas e que não houve perda de dados ao aplicar o schema pretendido, vamos começar a fazer um profiling inicial.

Nesta fase queremos perceber a estrutura e qualidade dos dados antes de definir regras de limpeza.


### 3.3.1 Contagens e schemas

In [0]:
# Faz-se uma contagem do nº de registos e de colunas de cada tabela na fase bronze.
# Neste caso, todas as tabelas foram carregadas com um schema que define todas as colunas como string.
# Ainda assim, é importante realizar uma última verificação.
for table, df in silver_dfs.items():
    print(f"Tabela {table.upper()} apresenta {bronze_dfs[table].count():,} registos.")
    print(f"Tabela {table.upper()} apresenta {len(bronze_dfs[table].columns)} colunas.")
    print(f"\nSchema da tabela {table.upper()}:")
    df.printSchema()


### 3.3.2 Verificação de nulos
Após a leitura das tabelas e a análise dos respetivos schemas, procede-se à verificação de valores nulos em cada coluna.

O objetivo desta etapa é avaliar a completude dos dados antes da aplicação das regras de limpeza da camada Silver. Para cada tabela, será calculado o número de valores nulos por coluna e a respetiva percentagem face ao total de registos.

Esta análise permite distinguir entre nulos em campos críticos, como `primaryid` e `caseid`, que podem comprometer a integridade relacional dos dados, e nulos em campos opcionais ou clinicamente informativos, cuja ausência pode ser esperada e deve ser preservada ou tratada com cautela.


In [0]:
# Reutilizar a função analisar_nulos() para profiling
print("=== ANÁLISE DETALHADA DE NULOS (Profiling) ===\n")

# Usar função para totais
_ = analisar_nulos(
    dfs_dict=silver_dfs,
    return_snapshots=False
)

# Análise detalhada por coluna
print("\n=== ANÁLISE DETALHADA POR COLUNA ===\n")

for table_name, df in silver_dfs.items():
    print(f"--- Tabela: {table_name.upper()} - Nulos por Coluna ---")
    
    cols = df.columns
    total_rows = df.count()
    
    # Contagem eficiente: uma única passagem
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in cols]
    null_counts_row = df.select(*null_exprs).collect()[0]
    
    # Construir lista para display
    null_data = []
    for col_name in cols:
        null_count = null_counts_row[col_name]
        null_pct = round((null_count / total_rows) * 100, 1) if total_rows > 0 else 0
        null_data.append((col_name, null_count, null_pct))
    
    # Criar DataFrame para display ordenado
    null_df = spark.createDataFrame(
        null_data,
        ["column_name", "null_count", "null_pct"]
    ).orderBy(F.desc("null_pct"))
    
    display(null_df)
    print()

### 3.2.3 Verificação de duplicados

A análise de duplicados será usada para definir a estratégia de limpeza na camada Silver.

Na camada Silver serão removidos duplicados exatos e serão mantidos os identificadores necessários para preservar relações entre tabelas. *A deduplicação por chave lógica será aplicada com cuidado, uma vez que algumas tabelas FAERS podem conter múltiplos registos válidos por caso, medicamento, reação ou desfecho.*


In [0]:
print("=== VERIFICAÇÃO DE DUPLICADOS EXATOS ===\n")

for table, df in bronze_dfs.items():
    print(f"--- Tabela: {table.upper()} ---")
    
    total_rows = df.count()
    distinct_rows = df.distinct().count()
    duplicate_rows = total_rows - distinct_rows
    
    print(f"  Total de registos: {total_rows:,}")
    print(f"  Registos distintos: {distinct_rows:,}")
    print(f"  Duplicados exatos: {duplicate_rows:,}")
    print()

### 3.2.4 Duplicados por chave lógica

Ao contrário dos duplicados exatos, onde todas as colunas da linha são iguais, os duplicados por chave lógica ocorrem quando existem várias linhas com a mesma combinação de campos identificadores. Estes casos podem representar duplicação real, mas também podem refletir a estrutura natural dos dados FAERS.

Nesta análise, não serão removidos registos automaticamente. O objetivo é apenas identificar situações em que a mesma chave lógica aparece mais do que uma vez, para compreender se esses casos exigem tratamento posterior na camada Silver.

As chaves lógicas consideradas são:

| Tabela | Chave lógica utilizada | Interpretação |
|---|---|---|
| `demo` | `primaryid`, `caseid`, `caseversion` | identifica uma versão específica de um caso |
| `drug` | `primaryid`, `caseid`, `drug_seq` | identifica um medicamento dentro de um caso |
| `reac` | `primaryid`, `caseid`, `pt` | identifica uma reação reportada num caso |
| `outc` | `primaryid`, `caseid`, `outc_cod` | identifica um outcome associado a um caso |

Esta análise permite perceber se existem combinações de chaves repetidas e avaliar se devem ser mantidas, investigadas ou removidas numa fase posterior. Em tabelas como `drug`, `reac` e `outc`, a existência de várias linhas por caso pode ser esperada, uma vez que um mesmo caso pode estar associado a múltiplos medicamentos, reações ou outcomes.

In [0]:
# Define um dicionário com as chaves lógicas a usar em cada tabela.
logical_keys = {
    "demo": ["primaryid", "caseid", "caseversion"],
    "drug": ["primaryid", "caseid", "drug_seq"],
    "reac": ["primaryid", "caseid", "pt"],
    "outc": ["primaryid", "caseid", "outc_cod"]
}

duplicate_keys_dfs = {}

# analise duplicados por chave lógica em cada tabela.
for table_name, key_cols in logical_keys.items():
    df = silver_dfs[table_name]

    duplicate_keys_dfs[table_name] = analisar_duplicados_chave_logica(
        df=df,
        table_name=table_name,
        key_cols=key_cols,
        show_results=True
    )

In [0]:
# Verificar se os duplicados por chave lógica são registos exatos ou únicos
print("=== INSPEÇÃO DE DUPLICADOS POR CHAVE LÓGICA ===\n")
print("Objetivo: Verificar se registos com a mesma chave lógica são idênticos (duplicados exatos)")
print("ou se são registos únicos que partilham apenas a chave.\n")

for table_name, key_cols in logical_keys.items():
    print(f"\n{'='*70}")
    print(f"Tabela: {table_name.upper()}")
    print(f"Chave lógica: {', '.join(key_cols)}")
    print(f"{'='*70}\n")
    
    df = silver_dfs[table_name]
    duplicate_keys_df = duplicate_keys_dfs[table_name]
    
    if duplicate_keys_df is None or duplicate_keys_df.count() == 0:
        print("✅ Não há duplicados por chave lógica nesta tabela.\n")
        continue
    
    # Pegar a primeira chave duplicada como exemplo
    first_duplicate_key = duplicate_keys_df.first()
    
    # Construir filtro para essa chave
    filter_condition = None
    for col in key_cols:
        if filter_condition is None:
            filter_condition = (F.col(col) == first_duplicate_key[col])
        else:
            filter_condition = filter_condition & (F.col(col) == first_duplicate_key[col])
    
    # Buscar todos os registos com essa chave
    duplicate_records = df.filter(filter_condition)
    num_duplicates = duplicate_records.count()
    
    print(f"Exemplo de chave duplicada:")
    for col in key_cols:
        print(f"  {col}: {first_duplicate_key[col]}")
    print(f"\nNúmero de registos com esta chave: {num_duplicates}")
    
    # Verificar se são duplicados exatos
    distinct_records = duplicate_records.distinct().count()
    
    if distinct_records == 1:
        print(f"\n⚠️  DUPLICADOS EXATOS: Todos os {num_duplicates} registos são idênticos.")
        print("   Ação recomendada: Remover duplicados com dropDuplicates().")
    else:
        print(f"\n✅ REGISTOS ÚNICOS: {distinct_records} registos distintos partilham a mesma chave.")
        print("   Ação recomendada: Manter registos (são dados válidos).")
    
    print(f"\nAmostra dos primeiros 20 registos:")
    display(duplicate_records.limit(20))
    print()

### 3.2.5 Verificação de ID

Nesta etapa verificamos se a coluna de identificação do dataset contém valores únicos para cada registo.

A unicidade do ID é importante porque em muitos datasets o identificador deve representar uma entidade ou observação única. Se existirem IDs repetidos, isso pode indicar duplicação de registos, erros de integração, problemas no carregamento dos dados ou situações específicas do domínio que precisam de ser analisadas.

A tabela `DEMO` contem o primaryID de todos os casos que constam em todas as tabelas, sendo que esta chave é concatenada do ID do Caso e do Número da Versão do Caso.

É possivel e aceitável que `primaryid` tenha duplicados noutras tabelas pois cada caso pode ter varias drogas/reações/outcomes associados


In [0]:
# Verificar se primaryid é único na tabela DEMO
print("=== VERIFICAÇÃO DE UNICIDADE DO primaryid NA DEMO ===\n")
print("primaryid deve ser a chave primária única só na tabela DEMO.\n")

df_demo = silver_dfs["demo"]

total_rows = df_demo.count()
distinct_primaryid = df_demo.select("primaryid").distinct().count()
duplicate_primaryid = total_rows - distinct_primaryid

print(f"  Total de registos: {total_rows:,}")
print(f"  primaryid distintos: {distinct_primaryid:,}")

if duplicate_primaryid > 0:
    print(f"  ⚠️  ATENÇÃO: {duplicate_primaryid:,} registos com primaryid duplicado!")
    print(f"     primaryid NÃO é único na tabela DEMO.")
else:
    print(f"  ✅ primaryid é único (sem duplicados)")

print()

### 3.2.6 Verificação de Datas Impossíveis

As datas são fundamentais para análises temporais no dataset FAERS (time-to-event, latência de reporte, trends). No entanto, erros de parsing, input manual incorreto, ou bugs nos sistemas fonte podem resultar em datas impossíveis que comprometem a qualidade dos dados.

Nesta fase de **profiling**, queremos identificar e quantificar a presença de datas fora dos limites aceitáveis **antes** de aplicar qualquer transformação.

#### **Datas a Verificar:**

**DEMO:**
* `event_dt` — Data do evento adverso
* `mfr_dt` — Data de receção pelo fabricante
* `init_fda_dt` — Data inicial de receção pela FDA
* `fda_dt` — Data de receção pela FDA (versão atual)
* `rept_dt` — Data de reporte

**DRUG:**
* `exp_dt` — Data de expiração do lote

---

### **Critérios de Validação**

✅ **Lower bound**: `1900-01-01`
* O sistema FAERS foi criado nos anos 60; qualquer data anterior a 1900 é claramente um erro

✅ **Upper bound**: Data atual (`current_date()`)
* Eventos reportados no futuro são impossíveis
* **Exceção**: `exp_dt` pode estar no futuro (datas de expiração válidas)

---

### **Objetivo desta Análise**

Identificar a **quantidade** e **percentagem** de datas inválidas em cada coluna para:
1. Perceber a magnitude do problema
2. Informar a estratégia de limpeza na camada Silver
3. Documentar a qualidade dos dados antes da transformação

In [0]:
# DEMO: Verificar 5 colunas de data com upper bound (eventos não podem estar no futuro)
df_demo = silver_dfs["demo"]

date_cols_demo = ["event_dt", "mfr_dt", "init_fda_dt", "fda_dt", "rept_dt"]

demo_date_analysis = analisar_datas_invalidas(
    df=df_demo,
    table_name="DEMO",
    date_cols=date_cols_demo,
    lower_bound="1900-01-01",
    upper_bound="2026-06-01",  # hoje
    show_results=True
)

In [0]:
# DRUG: Verificar exp_dt apenas com lower bound (datas de expiração podem estar no futuro)
df_drug = silver_dfs["drug"]

date_cols_drug = ["exp_dt"]

drug_date_analysis = analisar_datas_invalidas(
    df=df_drug,
    table_name="DRUG",
    date_cols=date_cols_drug,
    lower_bound="1900-01-01",
    upper_bound=None,  # Sem limite superior (expirações podem ser futuras)
    show_results=True
)

# Se houver datas inválidas, mostrar amostra
if drug_date_analysis is not None:
    invalid_count = drug_date_analysis.filter(F.col("total_inválidas") > 0).count()
    
    if invalid_count > 0:
        print("\n🔍 Amostra de datas inválidas:")
        lower_bound = F.lit("1900-01-01").cast("date")
        display(
            df_drug
            .filter(F.col("exp_dt") < lower_bound)
            .select("primaryid", "drug_seq", "drugname", "exp_dt")
            .limit(10)
        )

print("\n📌 REAC e OUTC não têm colunas de data")

### 3.2.7 Verificação de integridade de referência / deteção de entradas orfãs

Entradas órfãs nas tabelas filhas, isto é, registos com `primaryid` sem correspondência na tabela `DEMO`, não conseguem fazer join com a tabela principal na camada Gold.

Isto pode levar à perda desses registos nas análises finais ou à criação de resultados incompletos. 

Por isso, nesta fase verificamos se todos os `primaryid` presentes nas tabelas filhas existem também na `DEMO`.

In [0]:
# Verificar se todos os primaryid nas tabelas filhas existem na tabela DEMO
import pyspark.sql.functions as F

print("=== VERIFICAÇÃO DE INTEGRIDADE REFERENCIAL ===\n")
print("Verificar se todos os primaryid em DRUG, REAC e OUTC existem em DEMO.\n")

df_demo = silver_dfs["demo"]
valid_primaryids = df_demo.select("primaryid").distinct()

child_tables = ["drug", "reac", "outc"]

for table_name in child_tables:
    print(f"--- Tabela: {table_name.upper()} ---")
    
    df_child = silver_dfs[table_name]
    
    # Contar primaryids únicos na tabela filha
    total_primaryids = df_child.select("primaryid").distinct().count()
    
    # Encontrar primaryids órfãos (não existem em DEMO)
    orphan_primaryids = (
        df_child.select("primaryid")
        .distinct()
        .join(valid_primaryids, on="primaryid", how="left_anti")
    )
    
    orphan_count = orphan_primaryids.count()
    
    # Contar registos órfãos (não apenas IDs únicos)
    orphan_records = df_child.join(orphan_primaryids, on="primaryid", how="inner")
    orphan_records_count = orphan_records.count()
    
    print(f"  primaryid únicos na tabela: {total_primaryids:,}")
    print(f"  primaryid órfãos (não existem em DEMO): {orphan_count:,}")
    print(f"  Registos órfãos: {orphan_records_count:,}")
    
    if orphan_count > 0:
        print(f"  ⚠️  ATENÇÃO: Existem registos órfãos!")
        print(f"     Ação recomendada: Remover ou investigar estes registos.")
    else:
        print(f"  ✅ Integridade referencial OK (todos os primaryid existem em DEMO)")
    
    print()

### 3.2.8 Valores distintos em variáveis categóricas

Nesta etapa serão analisados os valores distintos existentes em algumas colunas categóricas relevantes das tabelas FAERS.

O objetivo é perceber que códigos e categorias aparecem nos dados antes da aplicação das regras de limpeza da camada Silver. Esta análise permite identificar inconsistências como diferenças de capitalização, espaços em branco, valores pouco frequentes, categorias desconhecidas ou códigos que possam necessitar de normalização.

Serão analisadas sobretudo colunas com significado categórico, como sexo, país do reporter, tipo de reporter, papel do medicamento no caso, via de administração, resultados clínicos e códigos de desfecho.

Para cada coluna selecionada, será apresentada a contagem de ocorrências por valor distinto, ordenada de forma decrescente. Esta informação será usada posteriormente para justificar transformações como `trim`, `upper` e o preenchimento de valores desconhecidos com códigos como `UNK` ou `U`.

In [0]:

categorical_cols = {
    "demo": ["sex", "occp_cod", "reporter_country", "e_sub", "wt_cod"],
    "drug": ["role_cod", "route", "dechal", "rechal", "dose_freq"],
    "reac": ["pt"],
    "outc": ["outc_cod"]
}

for table_name, cols in categorical_cols.items():
    print(f"--- Valores distintos em variáveis categóricas: {table_name.upper()} ---")
    
    df = silver_dfs[table_name]
    
    # OTIMIZADO: Computar schema uma vez antes do loop
    df_cols = df.columns
    
    for col_name in cols:
        if col_name not in df_cols:
            print(f"A coluna '{col_name}' não existe na tabela {table_name}.\n")
            continue
        
        print(f"Coluna: {col_name}")
        
        # Calcular total de registos para percentagem
        total_rows = df.count()
        
        distinct_values_df = (
            df.groupBy(col_name)
              .count()
              .withColumn("percentage", F.round((F.col("count") / total_rows) * 100, 2))
              .orderBy(F.desc("count"))
              .limit(30)
        )
        
        display(distinct_values_df)

### 3.3.2 Análise de Variáveis Numéricas

Após a análise de variáveis categóricas, é importante explorar as variáveis quantitativas para identificar:
- **Distribuições**: valores mínimos, máximos, médias e medianas.
- **Outliers**: valores extremos ou clinicamente implausíveis (ex: idades negativas, pesos acima de 500 kg).
- **Missing patterns**: verificar se existem padrões de ausência em doses ou métricas clínicas.

As principais variáveis numéricas nas tabelas FAERS são:

| Tabela | Variável | Descrição |
|--------|----------|------------|
| `demo` | `age` | Idade do paciente |
| `demo` | `wt` | Peso do paciente (kg) |
| `drug` | `dose_amt` | Quantidade da dose administrada |
| `drug` | `cum_dose_chr` | Dose cumulativa |

Esta análise ajudará a definir regras de limpeza para valores extremos e a decidir estratégias de imputação ou filtragem na camada Silver.

In [0]:
import pyspark.sql.functions as F

print("=== DISTRIBUIÇÃO DE CÓDIGOS DE UNIDADE ===\n")

# Analisar distribuição de age_cod na tabela demo
print("--- Tabela: DEMO | Coluna: age_cod ---")
df_demo = silver_dfs["demo"]

age_cod_dist = (
    df_demo.groupBy("age_cod")
    .agg(
        F.count("*").alias("count"),
        F.round((F.count("*") / df_demo.count()) * 100, 2).alias("percentage")
    )
    .orderBy(F.desc("count"))
)

display(age_cod_dist)

print("\nSignificado dos códigos de age_cod:")
print("  YR  = Anos (Years)")
print("  DY  = Dias (Days)")
print("  DEC = Décadas (Decades)")
print("  MON = Meses (Months)")
print("  WK  = Semanas (Weeks)")
print("  HR  = Horas (Hours)")

print("\n" + "-"*60 + "\n")

# Analisar distribuição de wt_cod na tabela demo
print("--- Tabela: DEMO | Coluna: wt_cod ---")

wt_cod_dist = (
    df_demo.groupBy("wt_cod")
    .agg(
        F.count("*").alias("count"),
        F.round((F.count("*") / df_demo.count()) * 100, 2).alias("percentage")
    )
    .orderBy(F.desc("count"))
)

display(wt_cod_dist)

print("\nSignificado dos códigos de wt_cod:")
print("  KG  = Quilogramas (Kilograms)")
print("  LBS = Libras (Pounds)")

In [0]:

# Definir colunas numéricas por tabela
numerical_cols = {
    "demo": ["age", "wt"],
    "drug": ["dose_amt", "cum_dose_chr"]
}

for table_name, cols in numerical_cols.items():
    print(f"\n{'='*60}")
    print(f"Análise de Variáveis Numéricas: {table_name.upper()}")
    print(f"{'='*60}\n")
    
    df = silver_dfs[table_name]
    
    for col_name in cols:
        if col_name not in df.columns:
            print(f"⚠️  Coluna '{col_name}' não existe na tabela {table_name}.\n")
            continue
        
        print(f"\n--- Coluna: {col_name.upper()} ---")
        
        # Para a coluna 'age', filtrar apenas registos onde age_cod = 'YR'
        # Para a coluna 'wt', filtrar apenas registos onde wt_cod = 'KG'
        if col_name == "age" and "age_cod" in df.columns:
            print("⚠️  Filtro aplicado: age_cod = 'YR' (apenas idades em anos)")
            df_filtered = df.filter(F.col("age_cod") == "YR")
        elif col_name == "wt" and "wt_cod" in df.columns:
            print("⚠️  Filtro aplicado: wt_cod = 'KG' (apenas pesos em quilogramas)")
            df_filtered = df.filter(F.col("wt_cod") == "KG")
        else:
            df_filtered = df
        
        # Estatísticas descritivas usando describe()
        stats_df = df_filtered.select(col_name).describe()
        display(stats_df)
        
        # Análise adicional: valores negativos e zeros
        total_rows = df.count()
        total_rows_filtered = df_filtered.count()
        negative_count = df_filtered.filter(F.col(col_name) < 0).count()
        zero_count = df_filtered.filter(F.col(col_name) == 0).count()
        null_count = df_filtered.filter(F.col(col_name).isNull()).count()
        
        print(f"\n📊 Análise de Qualidade:")
        if col_name in ["age", "wt"]:
            print(f"   Total de registos (tabela completa): {total_rows:,}")
            print(f"   Total de registos após filtro: {total_rows_filtered:,}")
        else:
            print(f"   Total de registos: {total_rows:,}")
        print(f"   Valores nulos: {null_count:,} ({round(null_count/total_rows_filtered*100, 2) if total_rows_filtered > 0 else 0}%)")
        print(f"   Valores negativos: {negative_count:,} ({round(negative_count/total_rows_filtered*100, 2) if total_rows_filtered > 0 else 0}%)")
        print(f"   Valores zero: {zero_count:,} ({round(zero_count/total_rows_filtered*100, 2) if total_rows_filtered > 0 else 0}%)")
        print(f"   Valores válidos (não-nulos e positivos): {total_rows_filtered - null_count - negative_count:,}")
        print("-" * 60)

In [0]:
print("\n" + "="*80)
print("ANÁLISE DE OUTLIERS - PERCENTIS E VALORES EXTREMOS")
print("="*80 + "\n")

# Analisar percentis para detectar outliers
numerical_cols = {
    "demo": ["age", "wt"],
    "drug": ["dose_amt", "cum_dose_chr"]
}

for table_name, cols in numerical_cols.items():
    print(f"\n--- Tabela: {table_name.upper()} ---\n")
    
    df = silver_dfs[table_name]
    
    for col_name in cols:
        if col_name not in df.columns:
            continue
        
        print(f"Coluna: {col_name.upper()}")
        
        # Para a coluna 'age', filtrar apenas registos onde age_cod = 'YR'
        # Para a coluna 'wt', filtrar apenas registos onde wt_cod = 'KG'
        if col_name == "age" and "age_cod" in df.columns:
            print("⚠️  Filtro aplicado: age_cod = 'YR' (apenas idades em anos)\n")
            df_filtered = df.filter(F.col("age_cod") == "YR")
        elif col_name == "wt" and "wt_cod" in df.columns:
            print("⚠️  Filtro aplicado: wt_cod = 'KG' (apenas pesos em quilogramas)\n")
            df_filtered = df.filter(F.col("wt_cod") == "KG")
        else:
            df_filtered = df
        
        # Calcular percentis (1%, 5%, 25%, 50%, 75%, 95%, 99%)
        percentiles = df_filtered.stat.approxQuantile(
            col_name, 
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99], 
            0.01
        )
        
        print(f"   P1:  {percentiles[0]}")
        print(f"   P5:  {percentiles[1]}")
        print(f"   P25: {percentiles[2]}")
        print(f"   P50 (Mediana): {percentiles[3]}")
        print(f"   P75: {percentiles[4]}")
        print(f"   P95: {percentiles[5]}")
        print(f"   P99: {percentiles[6]}")
        
        # Mostrar os 10 valores mais extremos (máximos)
        print(f"\n   Top 10 valores mais altos:")
        top_values = (
            df_filtered.select(col_name)
              .filter(F.col(col_name).isNotNull())
              .orderBy(F.desc(col_name))
              .limit(10)
        )
        display(top_values)
        
        print("-" * 60)

## 3.3.3 Síntese dos Resultados do Profiling

Após análise exaustiva das 4 tabelas FAERS (DEMO, DRUG, REAC, OUTC), foram identificados os seguintes padrões e problemas de qualidade de dados:

---

### 📊 **Estrutura e Volume**
* **DEMO**: 2,157,280 registos (tabela principal de casos)
* **DRUG**: 9,480,689 registos (múltiplos medicamentos por caso)
* **REAC**: 7,461,401 registos (múltiplas reações por caso)
* **OUTC**: 1,582,534 registos (outcomes de casos)

---

### ⚠️ **Problemas Identificados**

#### **1. Valores Nulos**
* **age/age_cod**: 45.48% null (impacta normalização de idade)
* **wt/wt_cod**: 83.48% null (maioria dos casos sem peso reportado)
* **Datas**: event_dt, mfr_dt com ~20-50% nulls

#### **2. Duplicados**
* **DRUG**: 4 duplicados exatos + 768 duplicados por chave lógica
* **REAC**: 108,888 duplicados exatos(85388 duplicados por chave lógica)

#### **3. Outliers Numéricos**
* **age**: Valores > 120 anos ou < 0 (clinicamente impossíveis)
* **wt**: Pesos > 500kg ou < 0.5kg (outliers extremos)
* **Ação**: Normalizar para unidades padrão e aplicar filtros durante normalização

#### **4. Inconsistências de Strings**
* Capitalização inconsistente: `"US"` vs `"us"` vs `"Us"`
* Espaços em branco: `" M "` vs `"M"`


#### **5. Datas Inválidas**
* Datas < 1900-01-01 (impossibílidade histórica)
* Datas futuras (eventos que ainda não ocorreram)


## 3.4 Data Cleaning (Transformações da Camada Silver)

Com base nos achados do profiling, aplicaremos agora as transformações necessárias para criar a camada Silver limpa e pronta para análise.

**Transformações a Aplicar:**
1. ✅ Tratamento de valores nulos (fill com "UNK" para categóricas)
2. ✅ Remoção de duplicados (window function + fda_dt)
3. ✅ Normalização de unidades (age_years, wt_kg com outlier removal)
4. ✅ Normalização de strings (trim + upper + empty → UNK)
5. ✅ Validação de datas (range check: 1900-01-01 a hoje)

**Importante:**
Todas as transformações são aplicadas ao dicionário `silver_dfs` em memória. A escrita final para Delta será feita na **Seção 3.5** após todas as transformações estarem concluídas.

### 3.4.1 Tratamento de Valores Nulos

O profiling revelou elevadas taxas de valores nulos em várias colunas:
* **age/age_cod**: 45.48% null
* **wt/wt_cod**: 83.48% null
* **event_dt**, **mfr_dt**: ~20-50% null
* **Colunas categóricas**: sex, occp_cod, reporter_country, e_sub, etc.

#### **Estratégia de Tratamento:**

**1. Variáveis Categóricas** → Preencher com `"UNK"` (unknown)
* Permite manter registos na análise
* Queries podem filtrar `WHERE col != 'UNK'` se necessário
* **Colunas afetadas**:
  * DEMO: sex, occp_cod, reporter_country, e_sub, occr_country, age_cod, wt_cod
  * DRUG: role_cod, route, dechal, rechal

**2. Variáveis Numéricas** → **Manter NULL**
* Imputação (ex: média) pode distorcer distribuições
* NULL indica explicitamente "valor não reportado"
* Queries podem filtrar facilmente com `WHERE col IS NOT NULL`
*  **Justificação**: Preservar NULL é mais transparente que criar colunas indicadoras redundantes
* **Colunas afetadas**: age, wt, dose_amt, cum_dose_chr

**3. Datas** → **Manter NULL**
* Datas ausentes são informação relevante (falta de reporte)
* **Colunas afetadas**: event_dt, mfr_dt, init_fda_dt, fda_dt, rept_dt, exp_dt

In [0]:


print("=" * 80)
print("1. TABELA DEMO - Tratamento de Nulls com Smart Fill")
print("=" * 80 + "\n")

df_demo = silver_dfs["demo"]
df_demo_before_fillna = silver_dfs["demo"]

# Criação de DataFrame com valores nulos
df_demo_clean = (
    df_demo
    #  validar M/F, caso contrário "UNK" (inclui nulls e valores inválidos)
    .withColumn(
        "sex",
        F.when(F.col("sex").isin("M", "F"), F.col("sex"))
         .otherwise("UNK")
    )
    # utilizar função coalesce para substituir nulls por "UNK" de forma mais simples
    .withColumn(
        "occp_cod",
        F.coalesce(F.col("occp_cod"), F.lit("UNK"))
    )
    .withColumn(
        "reporter_country",
        F.coalesce(F.col("reporter_country"), F.lit("UNK"))
    )
    .withColumn(
        "e_sub",
        F.coalesce(F.col("e_sub"), F.lit("UNK"))
    )
    .withColumn(
        "occr_country",
        F.coalesce(F.col("occr_country"), F.lit("UNK"))
    )
    .withColumn(
        "age_cod",
        F.coalesce(F.col("age_cod"), F.lit("UNK"))
    )
    .withColumn(
        "age_grp",
        F.coalesce(F.col("age_grp"), F.lit("UNK"))
    )
    .withColumn(
        "wt_cod",
        F.coalesce(F.col("wt_cod"), F.lit("UNK"))
    )
    .withColumn(
        "i_f_code",
        F.coalesce(F.col("i_f_code"), F.lit("UNK"))
    )
    .withColumn(
        "rept_cod",
        F.coalesce(F.col("rept_cod"), F.lit("UNK"))
    )
    .withColumn(
        "mfr_sndr",
        F.coalesce(F.col("mfr_sndr"), F.lit("UNK"))
    )
    .withColumn(
        "mfr_num",
        F.coalesce(F.col("mfr_num"), F.lit("UNK"))
    )
)

print("\n✓ Nulls preenchidos: sex, occp_cod, reporter_country, e_sub, occr_country, age_cod, age_grp, wt_cod")
print("✓ + i_f_code, rept_cod, mfr_sndr, mfr_num (metadata importantes)")
print("\n➡️  DataFrame limpo guardado em: df_demo_clean")

In [0]:
# verificação rápida de limpeza
print("📋 Amostra de registos após tratamento de nulls:\n")
display(
    df_demo_clean
    .select("primaryid", "sex", "age", "age_cod", "wt", "wt_cod")
    .limit(20)
)
print("💡 Verificar: colunas categóricas não devem ter NULLs, apenas 'UNK'")
print("\nQuantidade de nulos antes de limpeza de nulos:", df_demo_before_fillna.filter(F.col("sex").isNull()).count())
print("Quantidade de nulos depois de limpeza de nulos:", df_demo_clean.filter(F.col("sex").isNull()).count())
print("Quantidade de UNK antes de limpeza de nulos:", df_demo_before_fillna.filter(F.col("sex") == "UNK").count())
print("Quantidade de UNK depois de limpeza de nulos:", df_demo_clean.filter(F.col("sex") == "UNK").count())


In [0]:
print("=" * 80)
print("2. TABELA DRUG - Tratamento de Nulls")
print("=" * 80 + "\n")

df_drug = silver_dfs["drug"]

# Aplicar fillna (incluindo dose_freq e outras categóricas prioritárias)
df_drug_clean = df_drug.fillna({
    "role_cod": "UNK",
    "route": "UNK",
    "dechal": "UNK",
    "rechal": "UNK",
    "dose_freq": "UNK",      
    "prod_ai": "UNK",        
    "val_vbm": "UNK",        
    "dose_vbm": "UNK",       
    "dose_unit": "UNK",      
    "dose_form": "UNK"       
})

# validação rápida
print(f"\n✓ role_cod nulls antes: {silver_dfs['drug'].filter(F.col('role_cod').isNull()).count():,}")
print(f"✓ role_cod nulls depois: {df_drug_clean.filter(F.col('role_cod').isNull()).count():,}")
print(f"\n✓ dose_freq nulls antes: {silver_dfs['drug'].filter(F.col('dose_freq').isNull()).count():,}")
print(f"✓ dose_freq nulls depois: {df_drug_clean.filter(F.col('dose_freq').isNull()).count():,}")
print(f"\n✓ prod_ai nulls antes: {silver_dfs['drug'].filter(F.col('prod_ai').isNull()).count():,}")
print(f"✓ prod_ai nulls depois: {df_drug_clean.filter(F.col('prod_ai').isNull()).count():,}")
print("\n➡️  DataFrame limpo guardado em: df_drug_clean")

In [0]:
# atualizar dicionario com tabelas tratadas

print("=" * 80)
print("4. ATUALIZAR DICIONÁRIO SILVER_DFS")
print("=" * 80 + "\n")

print("📦 Atualizando dicionário silver_dfs com tabelas tratadas...\n")

silver_dfs["demo"] = df_demo_clean
silver_dfs["drug"] = df_drug_clean
# REAC e OUTC mantêm-se inalterados 

print("✅ Tabelas atualizadas no dicionário silver_dfs!")
print("\n💡 RESUMO:")
print("   • DEMO: nulls preenchidos em sex, occp_cod, reporter_country, e_sub, occr_country, age_cod, age_grp, wt_cod")
print("   • DRUG: nulls preenchidos em role_cod, route, dechal, rechal")
print("   • REAC: sem alterações (drug_rec_act tem 99.9% nulls)")
print("   • OUTC: sem alterações (zero nulls)")
print("   • Variáveis numéricas (age, wt, dose_amt): nulls MANTIDOS")

💡 RESUMO:

  • `DEMO`: nulls preenchidos em `sex`, `occp_cod`, `reporter_country`, `e_sub`, `occr_country`, `age_cod`, `wt_cod`, `i_f_code`, `rept_cod`, `mfr_sndr`, `mfr_num`;

  • `DRUG`: nulls preenchidos em `role_cod`, `route`, `dechal`, `rechal`, `dose_freq`, `prod_ai`, `val_vbm`, `dose_vbm`, `dose_unit`, `dose_form`;

  • `REAC`: sem alterações (`drug_rec_act` tem 99.9% nulos);

  • `OUTC`: sem alterações (zero nulos);

  • Variáveis numéricas (`age`, `wt`, `dose_amt`): **nulos mantidos**

### 3.4.2 Remoção de Duplicados com Window Function

O profiling revelou a presença de duplicados em **DRUG** e **REAC**:

* **DRUG**: Registos com a mesma chave lógica `(primaryid, caseid, drug_seq)` que diferem em dose, datas, ingredientes, ou outras colunas
* **REAC**: Registos com a mesma chave lógica `(primaryid, caseid, pt)` (preferred term) que diferem noutras colunas
* **DEMO e OUTC**: 0 duplicados detectados

---

### **Estratégia de Deduplicatão: HYBRID**

Utilizaremos **Window Functions** com abordagem **HYBRID** para balancear temporal correctness e data quality:

1. Particionar pelos campos da chave lógica
2. Ordenar por:
   * `fda_dt DESC` (versão mais recente primeiro) → **temporal correctness**
   * `completeness_score DESC` (mais completo como tiebreaker) → **data quality**
   * `caseversion DESC` (versão mais recente como tiebreaker final)
3. Atribuir `row_number() = 1` ao registo escolhido
4. Filtrar apenas `row_num = 1`

**Vantagens desta abordagem:**
* ✅ Remove todos os tipos de duplicados (exatos e por chave lógica) numa única operação
* ✅ Prioriza versão mais recente (alinhado com versioning FAERS)
* ✅ Usa completeness como tiebreaker (previne perda de dados quando versões têm mesma data)
* ✅ Lógica standardizada e reutilizável via função `deduplicate_hybrid()`
* ✅ Preserva integridade referencial (usa fda_dt da tabela DEMO)

---

### **Tabelas Afetadas:**
* **DEMO e OUTC**: Sem duplicados → copiar diretamente
* **DRUG**: Aplicar window function com `(primaryid, caseid, drug_seq)`
* **REAC**: Aplicar window function com `(primaryid, pt)`


In [0]:
# Inicializar dicionário para guardar DataFrames sem duplicados
dedup_dfs = {}


print("✅ Dicionário dedup_dfs inicializado")

In [0]:
# DEMO não tem duplicados, copiar diretamente
dedup_dfs["demo"] = silver_dfs["demo"]

In [0]:
# DRUG - Aplicar deduplicação HYBRID
df_drug = silver_dfs["drug"]
df_demo = silver_dfs["demo"]

# Aplicar função standardizada com estratégia HYBRID
df_drug_dedup = deduplicate_hybrid(
    df=df_drug,
    key_cols=["primaryid", "caseid", "drug_seq"],
    df_demo=df_demo,
    table_name="DRUG"
)

# Guardar no dicionário
dedup_dfs["drug"] = df_drug_dedup

In [0]:
# REAC - Aplicar deduplicação HYBRID
df_reac = silver_dfs["reac"]
df_demo = silver_dfs["demo"]

# Aplicar função standardizada com estratégia HYBRID
df_reac_dedup = deduplicate_hybrid(
    df=df_reac,
    key_cols=["primaryid", "caseid", "pt"],
    df_demo=df_demo,
    table_name="REAC"
)

# Guardar no dicionário
dedup_dfs["reac"] = df_reac_dedup

In [0]:
# OUTC não tem duplicados, copiar diretamente
dedup_dfs["outc"] = silver_dfs["outc"]


In [0]:
# Atualizar dicionário principal
silver_dfs = dedup_dfs
print("✅ Remoção de duplicados concluída!")


### 3.4.3 Normalização de Unidades e Criação de Age Groups

O profiling revelou que as variáveis `age` e `wt` na tabela **DEMO** são reportadas em múltiplas unidades, dificultando análises quantitativas diretas:

#### **Problema Identificado:**

**Age (Idade)**
* **52.99%** em Anos (`YR`)
* **1.02%** em Décadas (`DEC`)
* **0.46%** em Meses (`MON`)
* **0.05%** em Dias (`DY`), Semanas (`WK`), Horas (`HR`)
* **45.48%** NULL (sem idade reportada)

**Age Group (age_grp)**
* **74.5%** NULL - coluna original tem gaps críticos
* Muitos registos têm `age` mas não têm `age_grp`
* Necessário criar `age_grp_cleaned` derivada de `age_years` normalizada

**Weight (Peso)**
* **16.32%** em Kilogramas (`KG`)
* **0.21%** em Libras (`LBS`)
* **83.48%** NULL (sem peso reportado)

---

### **Estratégia de Normalização**

Criaremos **novas colunas normalizadas** com unidades padrão:
1. `age_years` → idade em **anos** (com validação de outliers)
2. `age_grp_cleaned` → **grupos etários FDA** derivados de `age_years`
3. `wt_kg` → peso em **kilogramas** (com validação de outliers)

**Fórmulas de Conversão - Age:**
```python
YR  → value * 1
DEC → value * 10
MON → value / 12
WK  → value / 52
DY  → value / 365
HR  → value / 8760
```

**Age Groups (FDA Standard):**
* `NEONATE`: < 1 mês (< 0.083 anos)
* `INFANT`: 1 mês - < 2 anos
* `CHILD`: 2 - < 12 anos
* `ADOLESCENT`: 12 - < 18 anos
* `ADULT`: 18 - < 65 anos
* `ELDERLY`: 65+ anos
* `UNK`: idade desconhecida (NULL)

**Fórmulas de Conversão - Weight:**
```python
KG  → value * 1
LBS → value * 0.453592
```

**Tratamento de Outliers:**
Durante a normalização, valores clinicamente impossíveis serão substituídos por NULL:
* `age_years`: < 0 ou > 120 → NULL
* `wt_kg`: < 0.5kg ou > 500kg → NULL

---

### **Vantagens:**
1. ✅ Análises quantitativas diretas (médias, distribuições)
2. ✅ Comparações entre casos válidas
3. ✅ Age groups consistentes para análise demográfica (reduz gap significativamente)
4. ✅ Preserva colunas originais (age, age_grp, wt, age_cod, wt_cod) para auditoria
5. ✅ Outliers removidos automaticamente durante normalização

In [0]:
print("=" * 80)
print("1. DEMO - Normalização de Idade (age → age_years)")
print("=" * 80 + "\n")

df_demo = silver_dfs["demo"]

# Criar coluna age_years com conversões de unidades
# Nota: age já é DoubleType após schema casting (cell 11), por isso não se espera erros devido aos calculos efetuados
df_demo_normalized = (
    df_demo
    .withColumn(
        "age_years",
        F.when(F.upper(F.col("age_cod")) == "YR", F.col("age"))
         .when(F.upper(F.col("age_cod")) == "DEC", F.col("age") * 10)
         .when(F.upper(F.col("age_cod")) == "MON", F.round(F.col("age") / 12, 1))
         .when(F.upper(F.col("age_cod")) == "DY", F.round(F.col("age") / 365.25, 2))
         .when(F.upper(F.col("age_cod")) == "WK", F.round(F.col("age") / 52.18, 2))
         .when(F.upper(F.col("age_cod")) == "HR", F.round(F.col("age") / 8766, 4))
         .otherwise(F.lit(None))
    )
    # Validação: idade deve estar entre 0 e 120 anos
    .withColumn(
        "age_years",
        F.when(F.col("age_years").between(0, 120), F.col("age_years"))
         .otherwise(F.lit(None))
    )
)

print("✓ Coluna age_years criada com conversão de unidades")
print("✓ Validação aplicada: 0 ≤ age_years ≤ 120")

# Mostrar amostra
print("\n📋 Amostra de conversões:")
display(
    df_demo_normalized
    .filter(F.col("age_years").isNotNull())
    .select("primaryid", "age", "age_cod", "age_years")
    .limit(50)
)

#### **Justificação: Criação de `age_grp_cleaned`**

Antes de terminarmos a normalização de idade, criaremos uma nova coluna **`age_grp_cleaned`** derivada de `age_years` normalizada.

**Por que precisamos dela?**

A coluna original `age_grp` tem **74.5% de valores NULL** — três quartos dos registos não têm grupo etário atribuído. No entanto, muitos destes registos têm `age` preenchida, o que nos permite derivar grupos etários consistentes.

**Solução:**

Como acabámos de criar `age_years` (idade normalizada em anos), podemos derivar grupos etários consistentes e completos usando os **padrões FDA** de categorização demográfica:

* **NEONATE**: < 1 mês (< 0.083 anos)
* **INFANT**: 1 mês - < 2 anos  
* **CHILD**: 2 - < 12 anos
* **ADOLESCENT**: 12 - < 18 anos
* **ADULT**: 18 - < 65 anos
* **ELDERLY**: 65+ anos
* **UNK**: idade desconhecida (quando `age_years` é NULL)

**Vantagens:**
* ✅ Reduz significativamente o gap de 74.5% da coluna `age_grp` original
* ✅ Consistência: derivada da mesma fonte (`age_years`) para todos os registos
* ✅ Segue padrões FDA de categorização demográfica
* ✅ Preserva a coluna `age_grp` original para auditoria e comparação
* ✅ Permite análises demográficas robustas por coorte etária 

In [0]:
print("\n" + "=" * 80)
print("1.5 DEMO - Criação de Age Groups Limpos (age_grp_cleaned)")
print("=" * 80 + "\n")

# Criar coluna age_grp_cleaned usando FDA demographic standards
df_demo_normalized = (
    df_demo_normalized
    .withColumn(
        "age_grp_cleaned",
        F.when(F.col("age_years").isNull(), F.lit("UNK"))                    # NULL age
         .when(F.col("age_years") < 0.083, F.lit("NEONATE"))                # < 1 mês (~0.083 anos)
         .when(F.col("age_years") < 2, F.lit("INFANT"))                     # 1 mês - < 2 anos
         .when(F.col("age_years") < 12, F.lit("CHILD"))                     # 2 - < 12 anos
         .when(F.col("age_years") < 18, F.lit("ADOLESCENT"))                # 12 - < 18 anos
         .when(F.col("age_years") < 65, F.lit("ADULT"))                     # 18 - < 65 anos
         .otherwise(F.lit("ELDERLY"))                                        # 65+ anos
    )
)

print("✓ Coluna age_grp_cleaned criada com grupos etários FDA")
print("✓ Categorias: NEONATE, INFANT, CHILD, ADOLESCENT, ADULT, ELDERLY, UNK")

# Mostrar distribuição dos grupos etários
print("\n📊 Distribuição de Age Groups:")
df_age_grp_dist = (
    df_demo_normalized
    .groupBy("age_grp_cleaned")
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round((F.col("count") / df_demo_normalized.count()) * 100, 2))
    .orderBy(F.desc("count"))
)
display(df_age_grp_dist)

# Amostra de comparação: age_grp original vs. age_grp_cleaned
print("\n🔍 Comparação: age_grp (original) vs. age_grp_cleaned (derivado)")
display(
    df_demo_normalized
    .select("primaryid", "age", "age_cod", "age_years", "age_grp", "age_grp_cleaned")
    .limit(50)
)

print("\n✅ age_grp_cleaned criado com sucesso!")
print("📈 Reduz significativamente o gap de 74.5% da coluna age_grp original")
print("🎯 Pronto para análise demográfica estratificada por coorte etária")

In [0]:
print("\n" + "=" * 80)
print("2. DEMO - Normalização de Peso (wt → wt_kg)")
print("=" * 80 + "\n")

# Criar coluna wt_kg com conversões de unidades
# Nota: wt já é DoubleType após schema casting (cell 11)
df_demo_normalized = (
    df_demo_normalized
    .withColumn(
        "wt_kg",
        F.when(F.upper(F.col("wt_cod")) == "KG", F.col("wt"))
         .when(F.upper(F.col("wt_cod")) == "LBS", F.round(F.col("wt") * 0.453592, 2))
         .otherwise(F.lit(None))
    )
    # Validação: peso deve estar entre 0.5 e 500 kg
    .withColumn(
        "wt_kg",
        F.when(F.col("wt_kg").between(0.5, 500), F.col("wt_kg"))
         .otherwise(F.lit(None))
    )
)

print("✓ Coluna wt_kg criada com conversão de unidades")
print("✓ Validação aplicada: 0.5 ≤ wt_kg ≤ 500")

# Mostrar amostra
print("\n📋 Amostra de conversões:")
display(
    df_demo_normalized
    .filter(F.col("wt_kg").isNotNull())
    .select("primaryid", "wt", "wt_cod", "wt_kg")
    .limit(50)
)

# Atualizar dicionário
silver_dfs["demo"] = df_demo_normalized

print("\n✅ Normalização de unidades concluída!")
print("💡 Colunas age_years e wt_kg adicionadas à tabela DEMO")

**Nota:**

As variáveis numéricas mais relevantes foram normalizadas sempre que existia uma unidade de medida comum e comparável. 

No entanto, colunas como `dose_amt`e `cum_dose_chr` não foram normalizadas uma vez que não existe uma unidade padrão aplicável a todas as medicações.

As doses terapêuticas variam consoante o medicamento, a unidade utilizada e a via de administração, pelo que uma normalização direta poderia introduzir interpretações incorretas nos dados.

### 3.4.4 Normalização de Strings Categóricas e Tratamento de Strings Vazias

Durante o profiling, observámos que as colunas categóricas (códigos, países, rotas de administração, etc.) podem conter inconsistências de formatação que dificultam agrupamentos e análises:

#### **Problemas Identificados:**
* **Case inconsistente**: `"US"` vs. `"us"` vs. `"Us"`
* **Espaços em branco**: `" M "` vs. `"M"` (leading/trailing spaces)
* **Strings vazias**: Após `trim()`, valores como `"   "` tornam-se `""`
* **Dificulta JOINs e GROUP BY**: Valores semanticamente iguais são tratados como distintos

---

### **Estratégia de Normalização**

Aplicaremos três transformações em simultâneo a todas as colunas categóricas:

✅ **`trim()`**: Remove espaços em branco no início e fim da string
✅ **`upper()`**: Converte todas as letras para maiúsculas
✅ **Empty string → `"UNK"`**: Strings vazias são convertidas para "UNK" (unknown)

---

### **Tabelas e Colunas Afetadas:**

* **DEMO**: `sex`, `occp_cod`, `reporter_country`, `e_sub`, `occr_country`, `age_cod`, `wt_cod`
* **DRUG**: `role_cod`, `route`, `dechal`, `rechal`
* **REAC**: `pt` (preferred term)
* **OUTC**: `outc_cod`

---

### **Vantagens:**
1. **Consistência total**: Todos os valores "desconhecidos" representados como `"UNK"`
2. **JOINs mais robustos**: Evita falhas de match por diferenças de case/espaços
3. **Queries mais simples**: Não é necessário testar `col == "" OR col IS NULL`
4. **Performance**: Transformações aplicadas numa única passagem pelos dados

In [0]:
print("=" * 80)
print("1. DEMO - Normalização de Strings + Empty Strings → UNK")
print("=" * 80 + "\n")

df_demo = silver_dfs["demo"]

# Colunas categóricas a normalizar (incluindo metadata importantes)
categorical_cols = ["sex", "occp_cod", "reporter_country", "e_sub", "occr_country", "age_cod", "age_grp", "age_grp_cleaned", "wt_cod", "i_f_code", "rept_cod", "mfr_sndr", "mfr_num"]

# OTIMIZADO: Usar .withColumns() em vez de loop com .withColumn()
string_transformations = {
    col_name: F.when(F.upper(F.trim(F.col(col_name))) == "", F.lit("UNK"))
             .otherwise(F.upper(F.trim(F.col(col_name))))
    for col_name in categorical_cols
}

df_demo_normalized = df_demo.withColumns(string_transformations)

print(f"✓ Normalização aplicada a {len(categorical_cols)} colunas categóricas")
print(f"✓ Transformações: trim() + upper() + empty string → 'UNK'")
print(f"⚡ Otimização: .withColumns() aplicado em lote (1 operação vs {len(categorical_cols)})")

# Amostra de verificação
print("\n📋 Amostra de valores normalizados:")
display(
    df_demo_normalized
    .select("primaryid", "sex", "age_grp", "age_grp_cleaned", "occp_cod", "reporter_country", "i_f_code", "rept_cod", "mfr_sndr")
    .limit(10)
)

# Atualizar dicionário
silver_dfs["demo"] = df_demo_normalized
print("\n✅ DEMO normalizado!")

In [0]:
print("\n" + "=" * 80)
print("2. DRUG - Normalização de Strings + Empty Strings → UNK")
print("=" * 80 + "\n")

df_drug = silver_dfs["drug"]

# Colunas categóricas a normalizar (inclui drugname, dose_freq e outras prioritárias)
categorical_cols = ["role_cod", "route", "dechal", "rechal", "drugname", "dose_freq", "prod_ai", "val_vbm", "dose_vbm", "dose_unit", "dose_form"]

# OTIMIZADO: Usar .withColumns() em vez de loop com .withColumn()
# Transformações: trim + upper + remove non-alphanumeric + empty → UNK
string_transformations = {
    col_name: F.when(
        F.regexp_replace(F.upper(F.trim(F.col(col_name))), "[^A-Z0-9]", "") == "",
        F.lit("UNK")
    ).otherwise(
        F.regexp_replace(F.upper(F.trim(F.col(col_name))), "[^A-Z0-9]", "")
    )
    for col_name in categorical_cols
}

df_drug_normalized = df_drug.withColumns(string_transformations)

print(f"✓ Normalização aplicada a {len(categorical_cols)} colunas categóricas")
print(f"✓ Transformações: trim() + upper() + remove non-alphanumeric + empty string → 'UNK'")
print(f"✓ Inclui: drugname, dose_freq (crítico para Q1), prod_ai, dose_unit, dose_form")
print(f"⚡ Otimização: .withColumns() aplicado em lote (1 operação vs {len(categorical_cols)})")

# Amostra de verificação
print("\n📋 Amostra de valores normalizados:")
display(
    df_drug_normalized
    .select("primaryid", "drug_seq", "drugname", "dose_freq", "prod_ai", "dose_unit", "dose_form", "role_cod", "route")
    .limit(10)
)

# Atualizar dicionário
silver_dfs["drug"] = df_drug_normalized
print("\n✅ DRUG normalizado!")

In [0]:
print("\n" + "=" * 80)
print("3. REAC - Normalização de Strings + Empty Strings → UNK")
print("=" * 80 + "\n")

df_reac = silver_dfs["reac"]

# OTIMIZADO: Usar .withColumns() para consistência (mesmo com 1 coluna)
df_reac_normalized = df_reac.withColumns({
    "pt": F.when(F.upper(F.trim(F.col("pt"))) == "", F.lit("UNK"))
         .otherwise(F.upper(F.trim(F.col("pt"))))
})

print(f"✓ Normalização aplicada à coluna 'pt' (preferred term)")
print(f"✓ Transformações: trim() + upper() + empty string → 'UNK'")

# Amostra de verificação
print("\n📋 Amostra de valores normalizados:")
display(
    df_reac_normalized
    .select("primaryid", "pt")
    .limit(10)
)

# Atualizar dicionário
silver_dfs["reac"] = df_reac_normalized
print("\n✅ REAC normalizado!")

In [0]:
print("\n" + "=" * 80)
print("4. OUTC - Normalização de Strings + Empty Strings → UNK")
print("=" * 80 + "\n")

df_outc = silver_dfs["outc"]

# OTIMIZADO: Usar .withColumns() para consistência (mesmo com 1 coluna)
df_outc_normalized = df_outc.withColumns({
    "outc_cod": F.when(F.upper(F.trim(F.col("outc_cod"))) == "", F.lit("UNK"))
               .otherwise(F.upper(F.trim(F.col("outc_cod"))))
})

print(f"✓ Normalização aplicada à coluna 'outc_cod'")
print(f"✓ Transformações: trim() + upper() + empty string → 'UNK'")

# Amostra de verificação
print("\n📋 Amostra de valores normalizados:")
display(
    df_outc_normalized
    .select("primaryid", "outc_cod")
    .limit(10)
)

# Atualizar dicionário
silver_dfs["outc"] = df_outc_normalized

print("\n✅ OUTC normalizado!")
print("\n" + "=" * 80)
print("🎉 Normalização de strings concluída para todas as 4 tabelas!")
print("=" * 80)

### 3.4.5 Verificação e Validação de Datas

As datas são fundamentais para análises temporais (trends, time-to-event, latência de reporte). No entanto, erros de parsing, input manual incorreto, ou bugs nos sistemas fonte podem resultar em datas inválidas que comprometem a qualidade analítica.

#### **Datas Presentes no Dataset:**

**DEMO (tabela demográfica):**
* `event_dt` — Data do evento adverso
* `mfr_dt` — Data de receção pelo fabricante
* `init_fda_dt` — Data inicial de receção pela FDA
* `fda_dt` — Data de receção pela FDA (versão atual)
* `rept_dt` — Data de reporte

**DRUG (medicações):**
* `exp_dt` — Data de expiração do lote

REAC e OUTC não têm colunas de data, por isso não será feita nenhuma verificação.

---

### **Regras de Validação**

Aplicaremos validações de range temporal para identificar e neutralizar datas impossíveis:

✅ **Lower bound**: `1900-01-01`
* O sistema FAERS foi criado nos anos 60; qualquer data anterior a 1900 é claramente um erro

✅ **Upper bound**: Data atual (`current_date()`)
* Eventos no futuro são impossíveis (exceto `exp_dt`, onde futuro é válido)

✅ **Ação para datas inválidas**: Substituir por `NULL`
* Preserva o registo mas marca a data como não confiável
* Permite queries com `WHERE date IS NOT NULL` para filtrar casos válidos

---

### **Implementação**


**Nota sobre `exp_dt`:**
Para datas de expiração, é normal que estejam no futuro. Aplicaremos apenas o lower bound (1900-01-01).

---

### **Vantagens:**
1. **Análises temporais robustas**: Time-series sem outliers temporais
2. **Transparência**: Datas inválidas marcadas explicitamente como NULL
3. **Queries simplificadas**: Filtro `WHERE date IS NOT NULL` garante qualidade
4. **Auditoria**: Podemos contar quantas datas foram invalidadas

In [0]:
print("=" * 80)
print("1. DEMO - Verificação de Datas")
print("=" * 80 + "\n")

df_demo = silver_dfs["demo"]

# Colunas de data a validar
date_cols = ["event_dt", "mfr_dt", "init_fda_dt", "fda_dt", "rept_dt"]

# Definir limites temporais
lower_bound = F.lit("1900-01-01").cast("date")
upper_bound = F.current_date()

print(f"✓ Lower bound: 1900-01-01")
print(f"✓ Upper bound: {upper_bound} (hoje)\n")

# Contar datas inválidas antes da validação
print("🔍 Contagem de datas inválidas ANTES da validação:\n")
for col_name in date_cols:
    invalid_count = df_demo.filter(
        F.col(col_name).isNotNull() & 
        ~F.col(col_name).between(lower_bound, upper_bound)
    ).count()
    print(f"   {col_name:15s}: {invalid_count:,} datas fora do range")

# OTIMIZADO: Usar .withColumns() em vez de loop com .withColumn()
date_transformations = {
    col_name: F.when(
        F.col(col_name).between(lower_bound, upper_bound),
        F.col(col_name)
    ).otherwise(F.lit(None))
    for col_name in date_cols
}

df_demo_validated = df_demo.withColumns(date_transformations)

print("\n✅ Validação aplicada: datas fora do range [1900-01-01, hoje] → NULL")
print("⚡ Otimização: .withColumns() aplicado em lote (1 operação vs 5)")

In [0]:
# PRIMEIRO: Atualizar dicionário com dados limpos
silver_dfs["demo"] = df_demo_validated

print("\n" + "=" * 80)
print("🔍 VALIDAÇÃO RÁPIDA - DEMO (após limpeza)")
print("=" * 80 + "\n")

print("Objetivo: Confirmar que TODAS as datas inválidas foram substituídas por NULL\n")

# DEPOIS: Validar os dados limpos
demo_validation = analisar_datas_invalidas(
    df=silver_dfs["demo"],  # Agora sim, dados limpos!
    table_name="DEMO (após limpeza)",
    date_cols=["event_dt", "mfr_dt", "init_fda_dt", "fda_dt", "rept_dt"],
    lower_bound="1900-01-01",
    upper_bound=F.current_date(),
    show_results=True
)

print("\n✅ Se todas as colunas mostram 0 datas inválidas, a limpeza foi bem-sucedida!")
print("✅ DEMO: datas validadas!")

In [0]:
print("\n" + "=" * 80)
print("2. DRUG - Verificação de Data de Expiração")
print("=" * 80 + "\n")

df_drug = silver_dfs["drug"]

# Para exp_dt, apenas lower bound (datas de expiração podem estar no futuro)
lower_bound = F.lit("1900-01-01").cast("date")

# Contar datas inválidas antes da validação
invalid_count = df_drug.filter(
    F.col("exp_dt").isNotNull() & 
    (F.col("exp_dt") < lower_bound)
).count()

print(f"🔍 Datas de expiração < 1900-01-01: {invalid_count:,}\n")

# Aplicar validação
df_drug_validated = df_drug.withColumn(
    "exp_dt",
    F.when(
        F.col("exp_dt") >= lower_bound,
        F.col("exp_dt")
    ).otherwise(F.lit(None))
)



In [0]:
# PRIMEIRO: Atualizar dicionário com dados limpos
silver_dfs["drug"] = df_drug_validated

print("\n" + "=" * 80)
print("🔍 VALIDAÇÃO RÁPIDA - DRUG (após limpeza)")
print("=" * 80 + "\n")

print("Objetivo: Confirmar que TODAS as datas inválidas foram substituídas por NULL\n")

# DEPOIS: Validar os dados limpos
drug_validation = analisar_datas_invalidas(
    df=silver_dfs["drug"],  # Agora sim, dados limpos!
    table_name="DRUG (após limpeza)",
    date_cols=["exp_dt"],
    lower_bound="1900-01-01",
    upper_bound=None,  # Expirações podem ser futuras
    show_results=True
)

print("\n✅ Se a coluna mostra 0 datas inválidas, a limpeza foi bem-sucedida!")
print("✅ DRUG: datas validadas!")

## 3.5 Gravação da Camada Silver e Validação Final

Após a conclusão de todas as transformações de limpeza (null handling, deduplication, normalization, string cleaning, date validation), chegou o momento de **persistir** as tabelas Silver finais no Unity Catalog.

### **O que foi aplicado:**
✅ Tratamento de nulos (categóricas → "UNK")
✅ Remoção de duplicados (window function + fda_dt)
✅ Normalização de unidades (age_years, wt_kg + outlier removal)
✅ Normalização de strings (trim + upper + empty → UNK)
✅ Validação de datas (range 1900-hoje)

### **Destino:**
As 4 tabelas (DEMO, DRUG, REAC, OUTC) serão escritas em formato **Delta Lake** no caminho:
```
/Volumes/main/default/faers_data/delta/silver/
```

### **Modo de Escrita:**
* `mode("overwrite")` → Substitui tabelas existentes
* `option("overwriteSchema", "true")` → Permite atualização de schema

---

Após a escrita, realizaremos uma **validação final** lendo diretamente dos ficheiros Delta para confirmar:
1. Schema casting persistido corretamente
2. Contagens de registos corretas
3. Dados limpos e prontos para a camada Gold

In [0]:
print("=" * 80)
print("💾 ESCRITA DA CAMADA SILVER PARA DELTA LAKE")
print("=" * 80 + "\n")

print("📁 Destino: /Volumes/main/default/faers_data/delta/silver/\n")

# Escrever cada tabela em formato Delta
for table_name, df in silver_dfs.items():
    output_path = f"{silver_delta_path}/{table_name}"
    
    print(f"⏳ A escrever tabela: {table_name.upper()}...")
    
    # OTIMIZADO: Computar schema antes da escrita (evitar múltiplas RPCs)
    total_cols = len(df.columns)
    
    (
        df.write
          .mode("overwrite")
          .format("delta")
          .option("overwriteSchema", "true")
          .save(output_path)
    )
    
    # Contar registos escritos
    total_rows = df.count()
    
    print(f"✅ {table_name.upper()} escrita com sucesso!")
    print(f"   → Registos: {total_rows:,}")
    print(f"   → Colunas: {total_cols}")
    print(f"   → Caminho: {output_path}\n")

print("=" * 80)
print("✅ TODAS AS TABELAS SILVER ESCRITAS COM SUCESSO!")
print("=" * 80)
print("\n👉 Próximo passo: Validação final lendo diretamente do Delta")
print("   (confirmar schema, contagens e qualidade dos dados)\n")

In [0]:
print("=" * 80)
print("🔍 VALIDAÇÃO FINAL - LEITURA DIRETA DO DELTA")
print("=" * 80 + "\n")

print("Objetivo: Confirmar que as tabelas foram escritas corretamente")
print("Verificar: schema, contagens, e amostra de dados limpos\n")

for table in tables:
    silver_path = f"{silver_delta_path}/{table}"
    
    print("="*80)
    print(f"📋 Tabela: {table.upper()}")
    print("="*80)
    
    # Ler diretamente do caminho Delta gravado
    df_silver = spark.read.format("delta").load(silver_path)
    
    # OTIMIZADO: Computar schema uma vez antes de usar (evitar múltiplas RPCs)
    total_cols = len(df_silver.columns)
    
    # 1. Contagem total de linhas salvas
    total_rows = df_silver.count()
    
    print(f"\n✅ Total de registos persistidos: {total_rows:,}")
    print(f"✅ Total de colunas: {total_cols}")
    
    # 2. Print do Schema para validar visualmente o Casting
    print("\n📋 Schema (confirmar tipos date, int, double):")
    df_silver.printSchema()
    
    # 3. Mostrar uma amostra rápida dos dados limpos
    print(f"\n📄 Amostra dos primeiros 10 registos de {table.upper()}:")
    display(df_silver.limit(10))
    
    print("\n")

print("=" * 80)
print("✅✅✅ VALIDAÇÃO FINAL CONCLUÍDA COM SUCESSO! ✅✅✅")
print("=" * 80)
print("\n🎉 Camada Silver está pronta para a camada Gold!")
print("\n📊 Próximas análises:")
print("   • Estatísticas descritivas (age_years, wt_kg)")
print("   • Análise temporal (event_dt, fda_dt)")
print("   • JOINs entre tabelas para análise integrada")
print("   • Construção de métricas e agregados para dashboards")